# 💊 Dose-Safe Translator
**Track:** AI + Languages · Veritas AI International High School Hackathon 2026

**Who it's for:** a pharmacist at the counter whose patient reads only Vietnamese, Tagalog,
Haitian Creole, Chinese or Spanish, and needs to understand the directions on an
over-the-counter medicine.

**What goes in → what comes out:** an English medicine instruction such as
*"do not take more than 6 caplets in 24 hours"*, typed or read from a **photo of the label** →
a translation, that translation turned back into English, and a ✅ / ⚠️ verdict saying whether a
person should double-check it.

**Our question:** free AI translators sometimes make dangerous mistakes with medicine
directions, and people often check a translation by translating it back into English.
**Does that back-translation check actually catch the dangerous mistakes, and can targeted
safety checks catch more?**

**This project is a prototype and is not medical advice.**

## How to run this notebook
1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
   The first run downloads the translation model (about 2.5 GB) and takes roughly 10–15 minutes.
2. In section 2, set `TEAM_LANGUAGE` to the language a teammate reads fluently.
3. Section 8 makes a rating sheet. Your teammate rates each translation; upload the sheet and
   re-run section 8 to get the real results (missed dangerous errors and false alarms).
4. The app in section 11 prints a public link ending in `.gradio.live`. Open it in a new tab or
   on a phone (it works while this notebook is running).

In [ ]:
# 1 · Setup ───────────────────────────────────────────────────────────────
import importlib.util
import os
import subprocess
import sys

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")


def ensure_packages(packages):
    """Install any missing packages (Colab already has most of them)."""
    missing = [pip_name for module, pip_name in packages if importlib.util.find_spec(module) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


ensure_packages([("transformers", "transformers"), ("sentencepiece", "sentencepiece"),
                 ("sacrebleu", "sacrebleu"), ("gradio", "gradio")])

import random
import re
import time
import unicodedata
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

import pandas as pd
import requests
import sacrebleu
import torch
from IPython.display import Markdown, display

QUICK = os.environ.get("DOSESAFE_QUICK") == "1"  # tiny run, only used for testing the notebook
DEVICE = os.environ.get("DOSESAFE_DEVICE") or ("cuda" if torch.cuda.is_available() else "cpu")
random.seed(0)
pd.set_option("display.max_colwidth", 140)
print(f"Running on: {DEVICE.upper()}" + ("  (quick test mode)" if QUICK else ""))
if DEVICE == "cpu" and not QUICK:
    print("Tip: no GPU found. In Colab use Runtime → Change runtime type → T4 GPU (much faster).")

In [ ]:
# 2 · Settings ────────────────────────────────────────────────────────────
MODEL_NAME = "facebook/nllb-200-distilled-600M"  # free translation model from Meta (200 languages)

# For each language: the model's language code, words that mean "not / don't", number words
# (so the checks know "dos" means 2), and words for "under / less" and "over / more" limits.
# "ambiguous" lists number words that are also everyday words ("una" = a, "năm" = year), so a
# new one is not reported as a new number; "ambiguous_digits" does the same for digits in set
# phrases (Japanese 1日 = per day). "not_negations" lists words that contain a negation
# without meaning "not" (Chinese 不适 = discomfort, Thai อย่าง = like). "negation_starts" and
# "negation_ends" cover languages that put "not" inside a longer word (French n'utilisez, Turkish
# kullanmayın = don't use). "cjk" means the language is written without spaces between words, so
# words are found by their characters. "lang" is the web page's language tag and "rtl" marks
# languages written right to left.
# When in doubt, a list leaves a word out: a missing word makes the app ask for a check (a false
# alarm), while a wrong word could let a dangerous translation through.
# To add a language, copy an entry and fill it in (ask a fluent speaker to check the words).
# "flores" is the code used by the FLORES+ test set in section 9 (optional).
LANGUAGES = {
    "Spanish": {
        "code": "spa_Latn", "flores": ["spa_Latn"], "lang": "es",
        "negations": ["no", "nunca", "jamás", "ni", "tampoco"],
        "numbers": {"cero": 0, "uno": 1, "una": 1, "un": 1, "dos": 2, "tres": 3, "cuatro": 4,
                    "cinco": 5, "seis": 6, "siete": 7, "ocho": 8, "nueve": 9, "diez": 10,
                    "once": 11, "doce": 12, "quince": 15, "veinte": 20, "veinticuatro": 24,
                    "treinta": 30, "media": 0.5, "medio": 0.5},
        "ambiguous": ["uno", "una", "un", "media", "medio"],
        "limits": {"under/less": ["menos", "menor", "menores", "bajo", "inferior", "inferiores"],
                   "over/more": ["más", "mayor", "mayores", "exceder", "exceda", "excede",
                                 "superior", "superiores", "máximo", "máxima"]},
    },
    "Vietnamese": {
        "code": "vie_Latn", "flores": ["vie_Latn"], "lang": "vi",
        "negations": ["không", "đừng", "chớ", "cấm"],
        "numbers": {"một": 1, "hai": 2, "ba": 3, "bốn": 4, "tư": 4, "năm": 5, "sáu": 6, "bảy": 7,
                    "tám": 8, "chín": 9, "mười": 10, "mười một": 11, "mười hai": 12,
                    "mười lăm": 15, "hai mươi": 20, "hai mươi bốn": 24, "hai mươi tư": 24,
                    "ba mươi": 30, "nửa": 0.5},
        "ambiguous": ["một", "năm", "tư"],
        "limits": {"under/less": ["dưới", "ít hơn", "nhỏ hơn", "chưa đủ", "chưa tới"],
                   "over/more": ["trên", "quá", "hơn", "tối đa", "vượt", "trở lên"]},
    },
    "Tagalog": {
        "code": "tgl_Latn", "flores": ["fil_Latn", "tgl_Latn"], "lang": "tl",
        "negations": ["huwag", "hindi", "wag", "di"],
        "numbers": {"isa": 1, "isang": 1, "dalawa": 2, "dalawang": 2, "tatlo": 3, "tatlong": 3,
                    "apat": 4, "lima": 5, "limang": 5, "anim": 6, "pito": 7, "pitong": 7,
                    "walo": 8, "walong": 8, "siyam": 9, "sampu": 10, "sampung": 10,
                    "labing-isa": 11, "labindalawa": 12, "labindalawang": 12, "labinlima": 15,
                    "dalawampu": 20, "dalawampu't apat": 24, "kalahati": 0.5},
        "ambiguous": ["isa", "isang"],
        "limits": {"under/less": ["ilalim", "ibaba", "wala pang", "pababa", "mas bata", "kulang",
                                  "mababa", "hindi pa"],
                   "over/more": ["higit", "mahigit", "pataas", "lampas", "lumampas", "sobra",
                                 "mas matanda", "pinakamarami", "hanggang"]},
    },
    "Haitian Creole": {
        "code": "hat_Latn", "flores": ["hat_Latn"], "lang": "ht",
        "negations": ["pa", "pinga", "jamè", "okenn"],
        # "yon" (a / one) was missing, so "pran yon grenn" (take one tablet) was flagged.
        # A native speaker should confirm "yon", "en" and "de" here.
        "numbers": {"youn": 1, "yon": 1, "en": 1, "de": 2, "twa": 3, "kat": 4, "senk": 5, "sis": 6,
                    "sèt": 7, "uit": 8, "nèf": 9, "dis": 10, "onz": 11, "douz": 12, "kenz": 15,
                    "ven": 20, "vennkat": 24, "trant": 30, "mwatye": 0.5, "demi": 0.5},
        "ambiguous": ["youn", "yon", "en", "de"],
        "limits": {"under/less": ["anba", "mwens", "pi piti", "pi jèn", "poko"],
                   "over/more": ["plis", "depase", "pi gran", "anwo", "maksimòm", "pase"]},
    },
    "Chinese (Simplified)": {
        "code": "zho_Hans", "flores": ["cmn_Hans", "zho_Hans"], "cjk": True, "lang": "zh-Hans",
        "negations": ["不", "勿", "别", "没", "禁"],
        "not_negations": ["不适", "不良", "不舒服", "不同"],  # discomfort, adverse, unwell, different
        "numbers": {"一": 1, "二": 2, "两": 2, "三": 3, "四": 4, "五": 5, "六": 6, "七": 7, "八": 8,
                    "九": 9, "十": 10, "十一": 11, "十二": 12, "十五": 15, "二十": 20,
                    "二十四": 24, "三十": 30, "半": 0.5},
        "ambiguous": ["一"],  # also in 一些 (some), 一起 (together), 一般 (usually)
        "limits": {"under/less": ["以下", "不满", "未满", "少于", "低于", "小于", "不到"],
                   "over/more": ["以上", "超过", "多于", "高于", "大于", "最多", "超出"]},
    },
    # ---- Added Nov 15, 2026. Not yet tested on real label instructions or checked by native
    # speakers: the app marks them "not tested yet".
    "Chinese (Traditional)": {
        "code": "zho_Hant", "flores": ["cmn_Hant", "zho_Hant"], "cjk": True, "lang": "zh-Hant",
        "negations": ["不", "勿", "別", "沒", "禁"],
        "not_negations": ["不適", "不良", "不舒服", "不同"],
        "numbers": {"一": 1, "二": 2, "兩": 2, "三": 3, "四": 4, "五": 5, "六": 6, "七": 7, "八": 8,
                    "九": 9, "十": 10, "十一": 11, "十二": 12, "十五": 15, "二十": 20,
                    "二十四": 24, "三十": 30, "半": 0.5},
        "ambiguous": ["一"],
        "limits": {"under/less": ["以下", "不滿", "未滿", "少於", "低於", "小於", "不到"],
                   "over/more": ["以上", "超過", "多於", "高於", "大於", "最多", "超出"]},
    },
    "Korean": {
        # Korean is matched by characters: "not" is part of the verb ending (복용하지 마십시오,
        # 초과하지 않도록). One-syllable number words are left out because they hide in other
        # words (세 = 3 and also "years old" in 12세).
        "code": "kor_Hang", "flores": ["kor_Hang"], "cjk": True, "lang": "ko",
        "negations": ["않", "마십시오", "마세요", "말아", "말고", "말 것", "못하", "금지", "없이",
                      "안 됩", "안됩", "안 돼", "안돼"],
        "numbers": {"하나": 1, "한 번": 1, "둘": 2, "두 번": 2, "셋": 3, "세 번": 3, "넷": 4,
                    "네 번": 4, "다섯": 5, "여섯": 6, "일곱": 7, "여덟": 8, "아홉": 9, "열두": 12,
                    "스물": 20, "스물네": 24, "서른": 30, "절반": 0.5},
        "ambiguous": ["하나", "한 번"],
        "ambiguous_digits": ["1일"],  # per day: 1일 3회 = 3 times a day
        "limits": {"under/less": ["미만", "이하", "적게", "보다 적", "아래"],
                   "over/more": ["초과", "이상", "넘", "많이", "보다 많", "최대"]},
    },
    "Japanese": {
        "code": "jpn_Jpan", "flores": ["jpn_Jpan"], "cjk": True, "lang": "ja",
        "negations": ["ない", "ません", "禁止", "不可", "せず"],
        "not_negations": ["危ない"],  # dangerous
        "numbers": {"一": 1, "二": 2, "三": 3, "四": 4, "五": 5, "六": 6, "七": 7, "八": 8, "九": 9,
                    "十": 10, "十一": 11, "十二": 12, "十五": 15, "二十": 20, "二十四": 24,
                    "三十": 30, "半": 0.5},
        "ambiguous": ["一"],  # also in 一般 (usually), 一緒 (together)
        "ambiguous_digits": ["1日"],  # per day: 1日3回 = 3 times a day
        "limits": {"under/less": ["未満", "以下", "少なく", "少ない", "下回", "より少"],
                   "over/more": ["以上", "超え", "超過", "より多", "多く", "最大", "上回"]},
    },
    "Thai": {
        # ห้า (5) and สาม (3) are left out of the numbers: they hide in ห้าม (do not) and
        # สามารถ (can).
        "code": "tha_Thai", "flores": ["tha_Thai"], "cjk": True, "lang": "th",
        "negations": ["ไม่", "ห้าม", "อย่า", "ปราศจาก"],
        "not_negations": ["อย่าง", "ไม่สบาย", "ห้ามิลลิลิตร", "ห้ามล."],  # like, unwell, 5 mL
        "numbers": {"หนึ่ง": 1, "สอง": 2, "สี่": 4, "หก": 6, "เจ็ด": 7, "แปด": 8, "เก้า": 9,
                    "สิบ": 10, "สิบสอง": 12, "สิบห้า": 15, "ยี่สิบ": 20, "ยี่สิบสี่": 24,
                    "สามสิบ": 30, "ครึ่ง": 0.5},
        "ambiguous": ["หนึ่ง", "หก"],
        "limits": {"under/less": ["ต่ำกว่า", "น้อยกว่า", "ไม่ถึง"],
                   "over/more": ["เกิน", "มากกว่า", "ขึ้นไป", "สูงสุด", "นานกว่า", "ยาวกว่า"]},
    },
    "Russian": {
        "code": "rus_Cyrl", "flores": ["rus_Cyrl"], "lang": "ru",
        "negations": ["не", "нельзя", "без", "ни", "никогда", "запрещено", "нет"],
        "numbers": {"один": 1, "одна": 1, "одну": 1, "одного": 1, "одной": 1, "два": 2, "две": 2,
                    "двух": 2, "три": 3, "трех": 3, "трёх": 3, "четыре": 4, "четырех": 4,
                    "четырёх": 4, "пять": 5, "пяти": 5, "шесть": 6, "шести": 6, "семь": 7,
                    "семи": 7, "восемь": 8, "восьми": 8, "девять": 9, "девяти": 9, "десять": 10,
                    "десяти": 10, "одиннадцать": 11, "двенадцать": 12, "двенадцати": 12,
                    "пятнадцать": 15, "пятнадцати": 15, "двадцать": 20, "двадцати": 20,
                    "двадцать четыре": 24, "двадцати четырех": 24, "двадцати четырёх": 24,
                    "тридцать": 30, "тридцати": 30, "половина": 0.5, "половину": 0.5},
        "ambiguous": ["один", "одна", "одну", "одного", "одной"],
        "limits": {"under/less": ["младше", "моложе", "меньше", "менее", "ниже", "до лет", "до года"],
                   "over/more": ["больше", "более", "старше", "выше", "превышать", "превышайте",
                                 "превышает", "свыше", "максимум", "максимальная", "максимальной"]},
    },
    "Ukrainian": {
        "code": "ukr_Cyrl", "flores": ["ukr_Cyrl"], "lang": "uk",
        "negations": ["не", "ні", "ніколи", "без", "заборонено", "немає"],
        "numbers": {"один": 1, "одна": 1, "одну": 1, "одного": 1, "два": 2, "дві": 2, "двох": 2,
                    "три": 3, "трьох": 3, "чотири": 4, "чотирьох": 4, "п'ять": 5, "п’ять": 5,
                    "п'яти": 5, "п’яти": 5, "шість": 6, "шести": 6, "сім": 7, "семи": 7,
                    "вісім": 8, "восьми": 8, "дев'ять": 9, "дев’ять": 9, "десять": 10,
                    "десяти": 10, "одинадцять": 11, "дванадцять": 12, "дванадцяти": 12,
                    "п'ятнадцять": 15, "п’ятнадцять": 15, "двадцять": 20, "двадцяти": 20,
                    "двадцять чотири": 24, "двадцяти чотирьох": 24, "тридцять": 30,
                    "тридцяти": 30, "половина": 0.5, "половину": 0.5},
        "ambiguous": ["один", "одна", "одну", "одного"],
        "limits": {"under/less": ["молодше", "молодших", "менше", "менш", "нижче", "до років", "до року"],
                   "over/more": ["більше", "більш", "старше", "старших", "вище", "перевищувати",
                                 "перевищуйте", "перевищує", "понад", "максимум", "максимальна"]},
    },
    "Polish": {
        "code": "pol_Latn", "flores": ["pol_Latn"], "lang": "pl",
        "negations": ["nie", "nigdy", "bez", "zakaz", "nic"],
        "numbers": {"jeden": 1, "jedna": 1, "jedną": 1, "jedno": 1, "jednej": 1, "jednego": 1,
                    "dwa": 2, "dwie": 2, "dwóch": 2, "trzy": 3, "trzech": 3, "cztery": 4,
                    "czterech": 4, "pięć": 5, "pięciu": 5, "sześć": 6, "sześciu": 6, "siedem": 7,
                    "siedmiu": 7, "osiem": 8, "ośmiu": 8, "dziewięć": 9, "dziewięciu": 9,
                    "dziesięć": 10, "dziesięciu": 10, "jedenaście": 11, "dwanaście": 12,
                    "dwunastu": 12, "piętnaście": 15, "piętnastu": 15, "dwadzieścia": 20,
                    "dwudziestu": 20, "dwadzieścia cztery": 24, "dwudziestu czterech": 24,
                    "trzydzieści": 30, "trzydziestu": 30, "pół": 0.5, "połowa": 0.5, "połowę": 0.5},
        "ambiguous": ["jeden", "jedna", "jedną", "jedno", "jednej", "jednego"],
        "limits": {"under/less": ["poniżej", "mniej", "młodsze", "młodsi", "młodszych", "niżej"],
                   "over/more": ["powyżej", "więcej", "dłużej", "starsze", "starsi", "starszych",
                                 "przekraczać", "przekraczaj", "przekracza", "maksymalnie",
                                 "maksymalna", "ponad"]},
    },
    "French": {
        "code": "fra_Latn", "flores": ["fra_Latn"], "lang": "fr",
        "negations": ["ne", "pas", "jamais", "aucun", "aucune", "sans", "ni", "non", "interdit"],
        "negation_starts": ["n'", "n’"],  # n'utilisez pas
        "numbers": {"un": 1, "une": 1, "deux": 2, "trois": 3, "quatre": 4, "cinq": 5, "six": 6,
                    "sept": 7, "huit": 8, "neuf": 9, "dix": 10, "onze": 11, "douze": 12,
                    "quinze": 15, "vingt": 20, "vingt-quatre": 24, "vingt quatre": 24,
                    "trente": 30, "demi": 0.5, "demie": 0.5, "moitié": 0.5},
        "ambiguous": ["un", "une", "neuf"],  # neuf = nine or new
        "limits": {"under/less": ["moins", "inférieur", "inférieure", "inférieurs", "inférieures",
                                  "sous", "en dessous"],
                   "over/more": ["plus", "supérieur", "supérieure", "supérieurs", "supérieures",
                                 "dépasser", "dépassez", "dépasse", "au-delà", "maximum", "maximale"]},
    },
    "Portuguese": {
        "code": "por_Latn", "flores": ["por_Latn"], "lang": "pt",
        "negations": ["não", "nunca", "sem", "nem", "nenhum", "nenhuma", "proibido"],
        "numbers": {"um": 1, "uma": 1, "dois": 2, "duas": 2, "três": 3, "quatro": 4, "cinco": 5,
                    "seis": 6, "sete": 7, "oito": 8, "nove": 9, "dez": 10, "onze": 11, "doze": 12,
                    "quinze": 15, "vinte": 20, "vinte e quatro": 24, "trinta": 30, "meio": 0.5,
                    "meia": 0.5, "metade": 0.5},
        "ambiguous": ["um", "uma", "meio", "meia"],
        "limits": {"under/less": ["menos", "menor", "menores", "abaixo", "inferior", "inferiores"],
                   "over/more": ["mais", "maior", "maiores", "acima", "exceder", "exceda", "excede",
                                 "ultrapassar", "ultrapasse", "superior", "superiores", "máximo",
                                 "máxima"]},
    },
    "Italian": {
        "code": "ita_Latn", "flores": ["ita_Latn"], "lang": "it",
        "negations": ["non", "mai", "senza", "né", "nessun", "nessuna", "nessuno", "vietato"],
        "numbers": {"uno": 1, "una": 1, "un": 1, "due": 2, "tre": 3, "quattro": 4, "cinque": 5,
                    "sei": 6, "sette": 7, "otto": 8, "nove": 9, "dieci": 10, "undici": 11,
                    "dodici": 12, "quindici": 15, "venti": 20, "ventiquattro": 24, "trenta": 30,
                    "mezzo": 0.5, "mezza": 0.5, "metà": 0.5},
        "ambiguous": ["uno", "una", "un", "sei"],  # sei = six or "you are"
        "limits": {"under/less": ["meno", "inferiore", "inferiori", "sotto", "minore", "minori"],
                   "over/more": ["più", "superiore", "superiori", "oltre", "superare", "superi",
                                 "supera", "massimo", "massima", "sopra", "maggiore", "maggiori"]},
    },
    "German": {
        "code": "deu_Latn", "flores": ["deu_Latn"], "lang": "de",
        "negations": ["nicht", "kein", "keine", "keinen", "keinem", "keiner", "keines", "nie",
                      "niemals", "ohne", "verboten"],
        "numbers": {"ein": 1, "eine": 1, "einen": 1, "einem": 1, "einer": 1, "eins": 1, "zwei": 2,
                    "drei": 3, "vier": 4, "fünf": 5, "sechs": 6, "sieben": 7, "acht": 8, "neun": 9,
                    "zehn": 10, "elf": 11, "zwölf": 12, "fünfzehn": 15, "zwanzig": 20,
                    "vierundzwanzig": 24, "dreißig": 30, "halb": 0.5, "halbe": 0.5, "halben": 0.5,
                    "hälfte": 0.5},
        "ambiguous": ["ein", "eine", "einen", "einem", "einer", "eins", "acht"],
        "limits": {"under/less": ["unter", "weniger", "jünger", "unterhalb", "niedriger"],
                   "over/more": ["über", "mehr", "älter", "länger", "ab", "überschreiten",
                                 "überschreitet", "maximal", "höchstens", "oberhalb"]},
    },
    "Arabic": {
        "code": "arb_Arab", "flores": ["arb_Arab"], "lang": "ar", "rtl": True,
        "negations": ["لا", "ولا", "فلا", "لم", "لن", "ليس", "ليست", "عدم", "وعدم", "بعدم", "بدون",
                      "ممنوع", "يمنع", "ألا"],
        "numbers": {"واحد": 1, "واحدة": 1, "اثنين": 2, "اثنان": 2, "اثنتين": 2, "مرتين": 2,
                    "ثلاث": 3, "ثلاثة": 3, "أربع": 4, "أربعة": 4, "اربع": 4, "اربعة": 4, "خمس": 5,
                    "خمسة": 5, "ست": 6, "ستة": 6, "سبع": 7, "سبعة": 7, "ثماني": 8, "ثمانية": 8,
                    "تسع": 9, "تسعة": 9, "عشر": 10, "عشرة": 10, "اثني عشر": 12, "اثنا عشر": 12,
                    "اثنتي عشرة": 12, "خمسة عشر": 15, "خمس عشرة": 15, "عشرين": 20, "عشرون": 20,
                    "أربع وعشرين": 24, "أربعة وعشرين": 24, "اربع وعشرين": 24, "ثلاثين": 30,
                    "ثلاثون": 30, "نصف": 0.5, "مرة": 1},
        "ambiguous": ["واحد", "واحدة", "ست", "مرة"],  # مرة = once or "time"
        "limits": {"under/less": ["أقل", "لأقل", "بأقل", "دون", "تحت", "أصغر", "الأصغر"],
                   "over/more": ["أكثر", "لأكثر", "بأكثر", "فأكثر", "وأكثر", "أطول", "فوق", "تتجاوز", "يتجاوز",
                                 "تجاوز", "تتعدى", "يزيد", "تزيد", "أكبر", "الأكبر", "أقصى",
                                 "الأقصى"]},
    },
    "Persian (Farsi)": {
        "code": "pes_Arab", "flores": ["pes_Arab"], "lang": "fa", "rtl": True,
        "negations": ["نه", "نکنید", "نکنند", "نکند", "نکن", "نخورید", "ننوشید", "ندهید", "نگیرید",
                      "نباید", "نیست", "نمی", "نشود", "نشوید", "ننمایید", "بدون", "ممنوع", "هرگز",
                      "هیچ"],
        "numbers": {"یک": 1, "دو": 2, "سه": 3, "چهار": 4, "پنج": 5, "شش": 6, "هفت": 7, "هشت": 8,
                    "نه": 9, "ده": 10, "یازده": 11, "دوازده": 12, "پانزده": 15, "بیست": 20,
                    "بیست و چهار": 24, "سی": 30, "نیم": 0.5, "نصف": 0.5},
        "ambiguous": ["یک", "نه", "سی"],  # نه = nine or "no"
        "limits": {"under/less": ["کمتر", "کم", "زیر", "کوچکتر", "پایین"],
                   "over/more": ["بیشتر", "بیش", "بالای", "بالاتر", "به بالا", "حداکثر", "تجاوز",
                                 "بزرگتر", "طولانی تر"]},
    },
    "Urdu": {
        "code": "urd_Arab", "flores": ["urd_Arab"], "lang": "ur", "rtl": True,
        "negations": ["نہیں", "نہ", "مت", "بغیر", "ممنوع", "ہرگز"],
        "numbers": {"ایک": 1, "دو": 2, "تین": 3, "چار": 4, "پانچ": 5, "چھ": 6, "سات": 7, "آٹھ": 8,
                    "نو": 9, "دس": 10, "گیارہ": 11, "بارہ": 12, "پندرہ": 15, "بیس": 20,
                    "چوبیس": 24, "تیس": 30, "آدھا": 0.5, "آدھی": 0.5},
        "ambiguous": ["ایک"],
        "limits": {"under/less": ["کم", "سے کم", "نیچے", "سے چھوٹے"],
                   "over/more": ["زیادہ", "سے زیادہ", "اوپر", "سے اوپر", "زائد", "سے بڑے"]},
    },
    "Hindi": {
        "code": "hin_Deva", "flores": ["hin_Deva"], "lang": "hi",
        "negations": ["नहीं", "न", "मत", "बिना", "निषेध", "वर्जित"],
        "numbers": {"एक": 1, "दो": 2, "तीन": 3, "चार": 4, "पांच": 5, "पाँच": 5, "छह": 6, "छः": 6,
                    "सात": 7, "आठ": 8, "नौ": 9, "दस": 10, "ग्यारह": 11, "बारह": 12, "पंद्रह": 15,
                    "पन्द्रह": 15, "बीस": 20, "चौबीस": 24, "तीस": 30, "आधा": 0.5, "आधी": 0.5},
        "ambiguous": ["एक", "दो"],  # दो = two or "give"
        "limits": {"under/less": ["कम", "से कम", "नीचे", "से नीचे", "से छोटे"],
                   "over/more": ["अधिक", "से अधिक", "ज़्यादा", "से ज़्यादा", "ज्यादा", "से ज्यादा",
                                 "ऊपर", "से ऊपर", "अधिकतम", "से बड़े"]},
    },
    "Bengali": {
        "code": "ben_Beng", "flores": ["ben_Beng"], "lang": "bn",
        "negations": ["না", "নয়", "নেই", "নি", "বিনা", "নিষেধ", "কখনই"],
        "numbers": {"এক": 1, "দুই": 2, "তিন": 3, "চার": 4, "পাঁচ": 5, "ছয়": 6, "সাত": 7, "আট": 8,
                    "নয়": 9, "দশ": 10, "এগারো": 11, "বারো": 12, "পনেরো": 15, "বিশ": 20,
                    "কুড়ি": 20, "চব্বিশ": 24, "ত্রিশ": 30, "আধা": 0.5, "অর্ধেক": 0.5, "একবার": 1, "দুবার": 2},
        "ambiguous": ["এক", "নয়", "একবার"],  # নয় = nine or "is not"
        "limits": {"under/less": ["কম", "এর কম", "নিচে", "নীচে", "অনূর্ধ্ব"],
                   "over/more": ["বেশি", "এর বেশি", "অধিক", "উপরে", "ঊর্ধ্বে", "সর্বোচ্চ", "অতিক্রম"]},
    },
    "Punjabi": {
        "code": "pan_Guru", "flores": ["pan_Guru"], "lang": "pa",
        "negations": ["ਨਾ", "ਨਹੀਂ", "ਬਿਨਾਂ", "ਬਿਨਾ", "ਮਨਾਹੀ"],
        "numbers": {"ਇੱਕ": 1, "ਇਕ": 1, "ਦੋ": 2, "ਤਿੰਨ": 3, "ਚਾਰ": 4, "ਪੰਜ": 5, "ਛੇ": 6, "ਸੱਤ": 7,
                    "ਅੱਠ": 8, "ਨੌਂ": 9, "ਨੌ": 9, "ਦਸ": 10, "ਗਿਆਰਾਂ": 11, "ਬਾਰਾਂ": 12, "ਪੰਦਰਾਂ": 15,
                    "ਵੀਹ": 20, "ਚੌਵੀ": 24, "ਤੀਹ": 30, "ਅੱਧਾ": 0.5, "ਅੱਧੀ": 0.5},
        "ambiguous": ["ਇੱਕ", "ਇਕ"],
        "limits": {"under/less": ["ਘੱਟ", "ਤੋਂ ਘੱਟ", "ਹੇਠਾਂ", "ਤੋਂ ਛੋਟੇ"],
                   "over/more": ["ਵੱਧ", "ਤੋਂ ਵੱਧ", "ਜ਼ਿਆਦਾ", "ਜਿਆਦਾ", "ਤੋਂ ਜ਼ਿਆਦਾ", "ਉੱਪਰ", "ਤੋਂ ਉੱਪਰ",
                                 "ਤੋਂ ਵੱਡੇ", "ਵੱਧ ਤੋਂ ਵੱਧ"]},
    },
    "Gujarati": {
        "code": "guj_Gujr", "flores": ["guj_Gujr"], "lang": "gu",
        "negations": ["ન", "નહીં", "નહિ", "નથી", "વગર", "વિના", "મનાઈ", "પ્રતિબંધ"],
        "numbers": {"એક": 1, "બે": 2, "ત્રણ": 3, "ચાર": 4, "પાંચ": 5, "છ": 6, "સાત": 7, "આઠ": 8,
                    "નવ": 9, "દસ": 10, "અગિયાર": 11, "બાર": 12, "પંદર": 15, "વીસ": 20, "ચોવીસ": 24,
                    "ત્રીસ": 30, "અડધી": 0.5, "અડધો": 0.5, "અડધું": 0.5},
        "ambiguous": ["એક"],
        "limits": {"under/less": ["ઓછું", "ઓછા", "ઓછી", "થી ઓછું", "થી ઓછા", "નીચે"],
                   "over/more": ["વધુ", "થી વધુ", "વધારે", "થી વધારે", "ઉપર", "થી ઉપર", "થી મોટા",
                                 "મહત્તમ"]},
    },
    "Indonesian": {
        "code": "ind_Latn", "flores": ["ind_Latn"], "lang": "id",
        "negations": ["tidak", "jangan", "janganlah", "bukan", "tanpa", "tak", "dilarang"],
        "numbers": {"satu": 1, "sekali": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "enam": 6,
                    "tujuh": 7, "delapan": 8, "sembilan": 9, "sepuluh": 10, "sebelas": 11,
                    "dua belas": 12, "lima belas": 15, "dua puluh": 20, "dua puluh empat": 24,
                    "tiga puluh": 30, "setengah": 0.5, "separuh": 0.5},
        "ambiguous": ["satu", "sekali"],  # sekali = once or "very"
        # "belum" (not yet) is an age limit here ("belum berusia 12 tahun"), not a "not".
        "limits": {"under/less": ["di bawah", "kurang", "kurang dari", "lebih muda", "lebih sedikit",
                                  "lebih rendah", "belum"],
                   "over/more": ["lebih", "lebih dari", "melebihi", "di atas", "ke atas", "maksimal",
                                 "maksimum", "lebih tua", "paling banyak"]},
    },
    "Malay": {
        "code": "zsm_Latn", "flores": ["zsm_Latn"], "lang": "ms",
        "negations": ["tidak", "jangan", "bukan", "tanpa", "tak", "dilarang"],
        "numbers": {"satu": 1, "sekali": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "enam": 6,
                    "tujuh": 7, "lapan": 8, "sembilan": 9, "sepuluh": 10, "sebelas": 11,
                    "dua belas": 12, "lima belas": 15, "dua puluh": 20, "dua puluh empat": 24,
                    "tiga puluh": 30, "setengah": 0.5, "separuh": 0.5},
        "ambiguous": ["satu", "sekali"],
        "limits": {"under/less": ["di bawah", "kurang", "kurang daripada", "kurang dari",
                                  "lebih muda", "lebih sedikit", "belum"],
                   "over/more": ["lebih", "lebih daripada", "lebih dari", "melebihi", "di atas",
                                 "ke atas", "maksimum", "lebih tua", "paling banyak"]},
    },
    "Turkish": {
        "code": "tur_Latn", "flores": ["tur_Latn"], "lang": "tr",
        "negations": ["değil", "değildir", "yok", "asla", "hiç", "hiçbir", "yasak", "yasaktır"],
        "negation_ends": ["mayın", "meyin", "mayınız", "meyiniz", "mamalı", "memeli", "mamalıdır",
                          "memelidir", "maz", "mez", "madan", "meden", "masın", "mesin", "mamak",
                          "memek"],  # kullanmayın = don't use, almadan = without taking
        "numbers": {"bir": 1, "iki": 2, "üç": 3, "dört": 4, "beş": 5, "altı": 6, "yedi": 7,
                    "sekiz": 8, "dokuz": 9, "on": 10, "on bir": 11, "on iki": 12, "on beş": 15,
                    "yirmi": 20, "yirmi dört": 24, "otuz": 30, "yarım": 0.5, "yarı": 0.5},
        "ambiguous": ["bir"],
        "limits": {"under/less": ["altında", "altındaki", "altındakiler", "küçük", "küçükler",
                                  "daha az", "az", "aşağı"],
                   "over/more": ["üzerinde", "üzerindeki", "üstünde", "üzeri", "üstü", "fazla",
                                 "daha fazla", "fazlası", "en fazla", "büyük", "maksimum", "uzun süre",
                                 "daha uzun"]},
    },
    "Greek": {
        "code": "ell_Grek", "flores": ["ell_Grek"], "lang": "el",
        "negations": ["μην", "μη", "δεν", "ποτέ", "χωρίς", "όχι", "απαγορεύεται"],
        "numbers": {"ένα": 1, "μία": 1, "μια": 1, "ένας": 1, "δύο": 2, "τρία": 3, "τρεις": 3,
                    "τέσσερα": 4, "τέσσερις": 4, "πέντε": 5, "έξι": 6, "επτά": 7, "εφτά": 7,
                    "οκτώ": 8, "οχτώ": 8, "εννέα": 9, "εννιά": 9, "δέκα": 10, "έντεκα": 11,
                    "δώδεκα": 12, "δεκαπέντε": 15, "είκοσι": 20, "είκοσι τέσσερα": 24,
                    "είκοσι τέσσερις": 24, "τριάντα": 30, "μισό": 0.5, "μισή": 0.5},
        "ambiguous": ["ένα", "μία", "μια", "ένας"],
        "limits": {"under/less": ["κάτω", "λιγότερο", "λιγότερα", "λιγότερες", "μικρότερα",
                                  "μικρότερης", "μικρότερων", "νεότερα"],
                   "over/more": ["πάνω", "άνω", "περισσότερο", "περισσότερα", "περισσότερες",
                                 "υπερβαίνετε", "υπερβαίνει", "υπέρβαση", "μέγιστη", "μέγιστο",
                                 "μεγαλύτερα", "μεγαλύτερης"]},
    },
    "Hebrew": {
        "code": "heb_Hebr", "flores": ["heb_Hebr"], "lang": "he", "rtl": True,
        "negations": ["לא", "אל", "אין", "ללא", "בלי", "אסור", "ולא", "שלא", "מבלי"],
        "numbers": {"אחד": 1, "אחת": 1, "שניים": 2, "שתיים": 2, "שני": 2, "שתי": 2, "שלוש": 3,
                    "שלושה": 3, "ארבע": 4, "ארבעה": 4, "חמש": 5, "חמישה": 5, "שש": 6, "שישה": 6,
                    "שבע": 7, "שבעה": 7, "שמונה": 8, "תשע": 9, "תשעה": 9, "עשר": 10, "עשרה": 10,
                    "שתים עשרה": 12, "שנים עשר": 12, "עשרים": 20, "עשרים וארבע": 24,
                    "עשרים וארבעה": 24, "שלושים": 30, "חצי": 0.5, "פעם": 1, "פעמיים": 2},
        "ambiguous": ["אחד", "אחת", "שני", "שתי", "פעם"],  # פעם = once or "time"  # שני = two or "second"
        "limits": {"under/less": ["מתחת", "פחות"],
                   "over/more": ["יותר", "מעל", "ומעלה", "לחרוג", "חריגה", "מקסימום"]},
    },
}

ENGLISH = {
    "code": "eng_Latn",
    "negations": ["not", "never", "no", "nor", "cannot", "don't", "dont", "doesn't"],
    "numbers": {"zero": 0, "one": 1, "once": 1, "two": 2, "twice": 2, "three": 3, "four": 4,
                "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10, "eleven": 11,
                "twelve": 12, "fifteen": 15, "twenty": 20, "twenty-four": 24, "twenty four": 24,
                "thirty": 30, "half": 0.5},
    "ambiguous": ["one", "once"],
    "limits": {"under/less": ["under", "below", "younger", "less", "fewer", "lower"],
               "over/more": ["over", "above", "older", "more", "exceed", "exceeds", "exceeding",
                             "maximum", "most", "longer"]},
}

WIDELY_SERVED = "Spanish"      # a language free translators handle well, for comparison
TEAM_LANGUAGE = "Vietnamese"   # ← change this to the language your teammate reads fluently
EVAL_LANGUAGES = list(dict.fromkeys([WIDELY_SERVED, TEAM_LANGUAGE]))

N_INSTRUCTIONS = 12 if QUICK else 150  # label instructions to test in each language
N_CANDIDATES = 3 if QUICK else 5       # translations to try in "safer mode"
BACK_SIMILARITY_MIN = 40               # back-translation similarity (0–100) below this is flagged
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

## 3 · Real medicine directions from US drug labels
We use **openFDA** to find over-the-counter labels for common medicines, then download each
label's original file from **DailyMed** (US National Library of Medicine). In those files every
instruction is its own bullet point, so each one becomes one test sentence. We keep the
**Directions** section and the **Do not use** warnings. The result is saved to
`data/otc_directions.csv`, so later runs don't need the internet for this step.

In [ ]:
# 3 · Download and clean the label instructions ───────────────────────────
INGREDIENTS = ["acetaminophen", "ibuprofen", "naproxen sodium", "aspirin", "diphenhydramine",
               "loratadine", "cetirizine", "fexofenadine", "dextromethorphan", "guaifenesin",
               "phenylephrine", "famotidine", "omeprazole", "loperamide", "bismuth subsalicylate",
               "calcium carbonate", "docusate", "sennosides", "polyethylene glycol 3350",
               "doxylamine", "meclizine", "hydrocortisone", "miconazole", "nicotine"]
LABELS_PER_INGREDIENT = 3
DATA_FILE = DATA_DIR / "otc_directions.csv"
HEADERS = {"User-Agent": "dose-safe-hackathon-prototype"}
HL7 = "{urn:hl7-org:v3}"
SECTIONS = {"34068-7": "Directions", "50570-1": "Do not use"}  # label section codes
# Bullet marks. Some labels use a symbol-font bullet stored as a private-use character (like U+F06E).
BULLETS = r"[•●▪■◦\ue000-\uf8ff]"


def find_otc_labels(ingredient, limit):
    """Ask openFDA for over-the-counter labels that contain this ingredient."""
    query = f'openfda.product_type:"HUMAN OTC DRUG" AND openfda.generic_name:"{ingredient}"'
    response = requests.get("https://api.fda.gov/drug/label.json",
                            params={"search": query, "limit": limit}, headers=HEADERS, timeout=30)
    if response.status_code != 200:
        return []
    return [(r["set_id"], (r.get("openfda", {}).get("brand_name") or [""])[0])
            for r in response.json().get("results", []) if r.get("set_id")]


def fetch_label(set_id):
    """Download one label's original XML file from DailyMed."""
    url = f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{set_id}.xml"
    for attempt in range(3):
        try:
            response = requests.get(url, headers=HEADERS, timeout=30)
            if response.status_code == 200:
                return ET.fromstring(response.content)
        except (requests.RequestException, ET.ParseError):
            pass
        time.sleep(1 + attempt)
    return None


def clean(text):
    text = unicodedata.normalize("NFKC", text).replace("⁄", "/")
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    text = re.sub(BULLETS, " ", text)
    return " ".join(text.split()).strip(" ;,-–—:")


def element_text(element, skip_lists=False):
    """All the text inside an XML element (optionally leaving out nested lists and tables)."""
    parts = [element.text or ""]
    for child in element:
        if not (skip_lists and child.tag in (f"{HL7}list", f"{HL7}table")):
            parts.append(element_text(child, skip_lists))
        parts.append(child.tail or "")
    return " ".join(" ".join(parts).split())


def child_lists(element):
    """Lists directly inside this element (not lists nested deeper inside those lists)."""
    found = []
    for child in element:
        if child.tag == f"{HL7}list":
            found.append(child)
        elif child.tag != f"{HL7}table":
            found.extend(child_lists(child))
    return found


def list_units(xml_list, prefix=""):
    """Turn a bulleted list into sentences, e.g. 'adults: take 2 caplets every 6 hours'."""
    units = []
    for item in xml_list.findall(f"{HL7}item"):
        own_text = clean(element_text(item, skip_lists=True))
        label = f"{prefix}: {own_text}" if prefix and own_text else (prefix or own_text)
        nested = child_lists(item)
        if nested:
            for sub_list in nested:
                units += list_units(sub_list, label)
        elif label:
            units.append(label)
    return units


def table_units(table):
    """Dosing tables: 'children 6-11 years: take 1 tablet every 4 hours'."""
    units = []
    for row in table.iter(f"{HL7}tr"):
        if any(cell.tag == f"{HL7}th" for cell in row):
            continue  # header row
        cells = [clean(element_text(cell)) for cell in row if cell.tag == f"{HL7}td"]
        cells = [c for c in cells if c]
        if len(cells) >= 2:
            units.append(f"{cells[0]}: {'; '.join(cells[1:])}")
        elif cells:
            units.append(cells[0])
    return units


def section_units(section, heading):
    units = []
    # A line like "Do not use if you have:" introduces the bullets after it, so we keep it as a lead-in.
    title = section.find(f"{HL7}title")
    title_text = clean(element_text(title)) if title is not None else ""
    lead = title_text if heading == "Do not use" and len(title_text.split()) > 3 else ""
    text_element = section.find(f"{HL7}text")
    if text_element is not None:
        for child in text_element:
            if child.tag == f"{HL7}list":
                units += list_units(child, lead)
            elif child.tag == f"{HL7}table":
                units += table_units(child)
            elif child.tag == f"{HL7}paragraph":
                own_text = element_text(child, skip_lists=True)
                inner_lists = child_lists(child)
                if inner_lists:  # "Do not use if you have <bullets>" in one paragraph
                    paragraph_lead = clean(own_text)
                    if paragraph_lead.lower() in ("", "directions", "do not use"):
                        paragraph_lead = lead
                    for xml_list in inner_lists:
                        units += list_units(xml_list, paragraph_lead)
                elif own_text.rstrip().endswith(":"):
                    lead = clean(own_text)
                else:
                    units += [clean(piece) for piece in re.split(BULLETS, own_text)]
                for table in child.iter(f"{HL7}table"):
                    units += table_units(table)
    for sub_section in section.findall(f"{HL7}component/{HL7}section"):
        code = sub_section.find(f"{HL7}code")
        if code is None or code.get("code") in (None, "42229-5"):  # only untitled sub-parts
            units += section_units(sub_section, heading)
    if heading == "Do not use":  # turn bullets like "if you have an ulcer" into full warnings
        continuations = ("if ", "with ", "for ", "in ", "on ", "unless ", "while ", "to ")

        def full_warning(u):
            # Only the first sentence decides whether the bullet already says "not": a later
            # "If you are not sure, ask a doctor" must not stop "Do not use" from being added.
            first_sentence = re.split(r"(?<=[.!?])\s+", u.strip(), maxsplit=1)[0].lower()
            if re.search(r"\b(not|never)\b|n't\b", first_sentence):
                return u
            if u[:1].islower() or u.lower().startswith(continuations):
                return f"Do not use {u}"
            return u

        units = [full_warning(u) for u in units if u]
    # drop lead-ins that lost their bullets, like "Do not use if you have"
    return [u for u in units if not re.search(r"(if you (have|are)|the following)$", u.lower())]


def build_dataset():
    rows, seen = [], set()
    for ingredient in INGREDIENTS:
        for set_id, brand in find_otc_labels(ingredient, LABELS_PER_INGREDIENT):
            root = fetch_label(set_id)
            time.sleep(0.2)  # be polite to the free government servers
            if root is None:
                continue
            for section in root.iter(f"{HL7}section"):
                code = section.find(f"{HL7}code")
                heading = SECTIONS.get(code.get("code") if code is not None else None)
                if not heading:
                    continue
                for unit in section_units(section, heading):
                    unit = clean(unit)
                    if 4 <= len(unit.split()) <= 60 and unit.lower() not in seen:
                        seen.add(unit.lower())
                        rows.append({"instruction": unit, "section": heading,
                                     "ingredient": ingredient, "brand": brand, "set_id": set_id})
    return pd.DataFrame(rows)


if DATA_FILE.exists():
    data = pd.read_csv(DATA_FILE)
    print(f"Loaded {len(data)} saved instructions from {DATA_FILE}")
else:
    print("Downloading labels from openFDA and DailyMed (about 1–2 minutes)...")
    data = build_dataset()
    if data.empty:
        raise RuntimeError("No label lines came back from openFDA/DailyMed, so nothing was saved. "
                           "Check the connection and run this cell again.")
    data.insert(0, "instruction_id", range(1, len(data) + 1))
    data.to_csv(DATA_FILE, index=False)
    print(f"Saved {len(data)} unique instructions from {data.set_id.nunique()} labels to {DATA_FILE}")
display(data.sample(min(8, len(data)), random_state=1)[["instruction", "section", "ingredient"]])

## 4 · Safety checks: how we spot a dangerous translation
A translation is **flagged** (⚠️ "ask a pharmacist to check") if any of these fail.
The first four look at the **translation itself**; the rest look at the **back-translation**
(the translation turned back into English).

| Check | What it looks for |
|---|---|
| **Numbers kept** | Every number in the English (2 caplets, 6 hours, 12 years) appears in the translation, as digits or words, and no new number appears. |
| **"Not" kept** | The translation has at least as many "not / don't" words as the English. |
| **Under / over kept** | If the English says *under* 12 or *no more than* 6, the translation still has a word for *under* or *more* (Spanish *menores de*, Vietnamese *dưới*, Chinese *以下*, …). |
| **Actually translated** | The model didn't just copy the English. |
| **Numbers survive the round trip** | Every number is still there after translating back into English. |
| **"Not" survives the round trip** | The back-translation still says "not" (or "without"). |
| **Units survive the round trip** | Hours stay hours, days stay days, mg stays mg. |
| **Under / over survives the round trip** | The back-translation still says *under*, *over*, *more*, *less*… |
| **Back-translation matches** | The back-translation is similar to the original (chrF score). This is the check people usually do by eye. |

These are simple word rules, so they can't understand meaning. That is exactly why section 8
tests them against a real speaker's judgement.

In [ ]:
# 4 · Safety checks ─────────────────────────────────────────────────────────
DIGITS = re.compile(r"\d+(?:[.,]\d+)?(?:\s*/\s*\d+)?")
# A word is letters plus the combining marks many scripts write vowels with (Hindi नहीं, Thai ไม่).
# Python's \w leaves those marks out, which cut such words into pieces at every vowel sign.
MARKS = "".join(chr(c) for c in range(0x300, 0x10000) if unicodedata.category(chr(c)).startswith("M"))
LETTER = rf"(?:[^\W\d_]|[{MARKS}])"
WORDS = re.compile(rf"{LETTER}+(?:['’-]{LETTER}+)*")
UNITS = {  # ages ("12 years") are already protected by the numbers check
    "hours": ["hour", "hours", "hr", "hrs", "hourly"],
    "days": ["day", "days", "daily"],
    "weeks": ["week", "weeks", "weekly"],
    "months": ["month", "months", "monthly"],
    "minutes": ["minute", "minutes", "min", "mins"],
    "mg": ["mg", "milligram", "milligrams"],
    "mL": ["ml", "milliliter", "milliliters", "millilitre", "millilitres"],
    "teaspoons": ["teaspoon", "teaspoons", "tsp"],
    "tablespoons": ["tablespoon", "tablespoons", "tbsp"],
}
FORWARD_CHECKS = ["Numbers kept", "“Not” kept", "Under / over kept", "Actually translated"]
ROUND_TRIP_CHECKS = ["Numbers survive the round trip", "“Not” survives the round trip",
                     "Units survive the round trip", "Under / over survives the round trip",
                     "Back-translation matches"]
CHECK_NAMES = FORWARD_CHECKS + ROUND_TRIP_CHECKS

def normalize(text):
    # Turkish "İ" lowercases to "i" plus a combining dot; drop the dot so "İlaç" matches "ilaç".
    text = unicodedata.normalize("NFKC", text).replace("⁄", "/").lower().replace("i̇", "i")
    # The model sometimes writes "don 't" with a space; join it so the "not" still counts.
    return re.sub(r"(\w)\s+(['’]t)\b", r"\1\2", text)


def match_phrases(text, phrases, cjk=False):
    """Find phrases in a normalized text, trying the longest phrase first at each position, so each
    word belongs to one phrase only ("ít hơn" = less than is not also read as "hơn" = more).

    `phrases` maps each phrase to a label. Returns (the labels found, in order; what is left of the
    text: a string for Chinese, a list of words otherwise)."""
    found = []
    if cjk:  # no spaces between words (Chinese, Japanese, Thai): match characters
        table = {}
        for phrase, label in phrases.items():  # normalized like the text (Thai ำ changes)
            table.setdefault(normalize(phrase), label)
        keys = sorted((k for k in table if k), key=len, reverse=True)
        rest, i = list(text), 0
        while i < len(text):
            match = next((k for k in keys if text.startswith(k, i)), None)
            if match:
                found.append(table[match])
                rest[i:i + len(match)] = " " * len(match)
                i += len(match)
            else:
                i += 1
        return found, "".join(rest)
    table = {tuple(WORDS.findall(normalize(p))): label for p, label in phrases.items()}
    table.pop((), None)
    tokens, rest, i = WORDS.findall(text), [], 0
    longest = max((len(k) for k in table), default=1)
    while i < len(tokens):
        for size in range(min(longest, len(tokens) - i), 0, -1):
            key = tuple(tokens[i:i + size])
            if key in table:
                found.append(table[key])
                i += size
                break
        else:
            rest.append(tokens[i])
            i += 1
    return found, rest


def digit_numbers(text):
    """Numbers written as digits, e.g. "2", "7.5", "1/2" (= 0.5)."""
    digits = []
    for token in DIGITS.findall(normalize(text)):
        token = token.replace(" ", "")
        try:
            if "/" in token:
                top, bottom = token.split("/")
                digits.append(round(float(top.replace(",", ".")) / float(bottom), 3))
            else:
                digits.append(round(float(token.replace(",", ".")), 3))
        except (ValueError, ZeroDivisionError):
            pass
    return digits


def word_numbers(text, spec):
    """Numbers written as words, as (word, value) pairs, e.g. [("dos", 2)]."""
    words, _ = match_phrases(normalize(text), {w: (w, v) for w, v in spec["numbers"].items()},
                             spec.get("cjk"))
    return words


def number_parts(text, spec):
    """Numbers written as digits, and numbers written as words, in a text."""
    return digit_numbers(text), [float(v) for _, v in word_numbers(text, spec)]


def count_negations(text, spec):
    """How many "not / don't" words a text has. Words that only contain a negation (Chinese 不满 =
    under, 不适 = discomfort; Tagalog "hindi pa" = not yet, as in an age limit) don't count.
    Some languages say "not" inside a longer word: "negation_starts" and "negation_ends" say how
    such words start or end (French n'utilisez, Turkish kullanmayın = don't use)."""
    skip = [p for phrases in spec["limits"].values() for p in phrases] + spec.get("not_negations", [])
    _, rest = match_phrases(normalize(text), dict.fromkeys(skip), spec.get("cjk"))
    if spec.get("cjk"):
        return sum(rest.count(normalize(word)) for word in spec["negations"])
    negations = {normalize(word) for word in spec["negations"]}
    starts = tuple(normalize(word) for word in spec.get("negation_starts", []))
    ends = ("n't", "n’t") + tuple(normalize(word) for word in spec.get("negation_ends", []))
    return sum(1 for t in rest if t in negations or t.endswith(ends) or t.startswith(starts))


def word_groups(text, groups):
    tokens = set(WORDS.findall(normalize(text)))
    return {name for name, words in groups.items() if tokens & set(words)}


def limit_groups(text, spec):
    """Which kinds of limit ("under/less", "over/more") a text mentions, in its own language."""
    phrases = {p: name for name, words in spec["limits"].items() for p in words}
    found, _ = match_phrases(normalize(text), phrases, spec.get("cjk"))
    return set(found)


def _show(values):
    return ", ".join(str(int(v)) if float(v).is_integer() else str(v) for v in sorted(values))


def compare_numbers(source, text, spec):
    src_digits, src_words = number_parts(source, ENGLISH)
    needed = Counter(src_digits + src_words)
    digits, word_hits = digit_numbers(text), word_numbers(text, spec)
    missing = needed - Counter(digits + [float(v) for _, v in word_hits])
    # A number that appears out of nowhere is a problem, written as digits or as a word. Words that
    # are also ordinary words ("una" = a, "năm" = year) can't be trusted as new numbers, and nor can
    # digits in set phrases ("ambiguous_digits": Japanese 1日 = per day, as in 1日3回 = 3 times a day).
    ambiguous = set(spec.get("ambiguous", []))
    firm_words = [float(v) for w, v in word_hits if w not in ambiguous]
    set_phrases = Counter()
    for phrase in spec.get("ambiguous_digits", []):
        set_phrases.update(digit_numbers(phrase) * normalize(text).count(normalize(phrase)))
    firm_digits = list((Counter(digits) - set_phrases).elements())
    new = Counter(firm_digits + firm_words) - needed
    problems = []
    if missing:
        problems.append("missing " + _show(missing.elements()))
    if new:
        problems.append("new " + _show(new.elements()))
    return not problems, "; ".join(problems)


def safety_checks(source, translation, back, spec, back_similarity_min=BACK_SIMILARITY_MIN):
    """Run every check. Returns {check name: {"ok": True/False, "detail": "..."}}."""
    results = {}
    n_not = count_negations(source, ENGLISH)
    source_limits = limit_groups(source, ENGLISH)

    # Checks on the translation itself
    ok, detail = compare_numbers(source, translation, spec)
    results["Numbers kept"] = {"ok": ok, "detail": detail}
    n = count_negations(translation, spec)
    results["“Not” kept"] = {"ok": n >= n_not, "detail": f"English has {n_not}, translation has {n}"}
    lost = source_limits - limit_groups(translation, spec)
    results["Under / over kept"] = {"ok": not lost, "detail": "no word for " + ", ".join(sorted(lost))}
    copied = sacrebleu.sentence_chrf(translation, [source]).score
    results["Actually translated"] = {"ok": copied < 75, "detail": f"{copied:.0f}/100 identical to English"}

    # Checks on the back-translation (the translation turned back into English)
    ok, detail = compare_numbers(source, back, ENGLISH)
    results["Numbers survive the round trip"] = {"ok": ok, "detail": detail}
    n = count_negations(back, ENGLISH) + len(re.findall(r"\bwithout\b", normalize(back)))
    results["“Not” survives the round trip"] = {"ok": n >= n_not,
                                                "detail": f"English has {n_not}, back-translation has {n}"}
    lost = word_groups(source, UNITS) - word_groups(back, UNITS)
    results["Units survive the round trip"] = {"ok": not lost, "detail": "lost " + ", ".join(sorted(lost))}
    lost = source_limits - limit_groups(back, ENGLISH)
    results["Under / over survives the round trip"] = {"ok": not lost,
                                                       "detail": "lost " + ", ".join(sorted(lost))}
    similarity = sacrebleu.sentence_chrf(back, [source]).score
    results["Back-translation matches"] = {"ok": similarity >= back_similarity_min,
                                           "detail": f"similarity {similarity:.0f}/100"}
    return results


def failed_checks(source, translation, back, spec):
    return [name for name, c in safety_checks(source, translation, back, spec).items() if not c["ok"]]


# Quick self-test with hand-made Spanish examples (translation, back-translation)
_src = "do not take more than 6 caplets in 24 hours"
_tests = {
    "correct": ("no tome más de 6 cápsulas en 24 horas", "do not take more than 6 capsules in 24 hours"),
    "numbers as words": ("no tome más de seis cápsulas en veinticuatro horas",
                         "do not take more than six capsules in twenty-four hours"),
    "lost the 'not'": ("tome más de 6 cápsulas en 24 horas", "take more than 6 capsules in 24 hours"),
    "wrong number": ("no tome más de 8 cápsulas en 24 horas", "do not take more than 8 capsules in 24 hours"),
    "hours → days": ("no tome más de 6 cápsulas en 24 días", "do not take more than 6 capsules in 24 days"),
}
for _name, (_translation, _back) in _tests.items():
    _failed = failed_checks(_src, _translation, _back, LANGUAGES["Spanish"])
    print(f"{_name:17} → " + ("passes" if not _failed else "FLAGGED: " + ", ".join(_failed)))
assert not failed_checks(_src, *_tests["correct"], LANGUAGES["Spanish"])
assert not failed_checks(_src, *_tests["numbers as words"], LANGUAGES["Spanish"])
assert all(failed_checks(_src, *_tests[k], LANGUAGES["Spanish"]) for k in ["lost the 'not'", "wrong number", "hours → days"])

_src = "children under 12 years: ask a doctor"
_correct = ("niños menores de 12 años: consulte a un médico", "children under 12 years: see a doctor")
_dropped = ("niños de 12 años: consulte a un médico", "children 12 years old: see a doctor")
print(f"{'under kept':17} → " + (", ".join(failed_checks(_src, *_correct, LANGUAGES["Spanish"])) or "passes"))
print(f"{'under dropped':17} → FLAGGED: " + ", ".join(failed_checks(_src, *_dropped, LANGUAGES["Spanish"])))
assert not failed_checks(_src, *_correct, LANGUAGES["Spanish"])
assert "Under / over kept" in failed_checks(_src, *_dropped, LANGUAGES["Spanish"])

# Bugs found in the code review (each let a dangerous translation through before the fix)
_src = "do not take more than 6 caplets in 24 hours"
assert "Under / over kept" in failed_checks(_src, "không uống ít hơn 6 viên trong 24 giờ", _src, LANGUAGES["Vietnamese"])
assert "“Not” kept" in failed_checks("do not give to children under 12 years", "可给不满12岁的儿童",
                                    "do not give to children under 12 years", LANGUAGES["Chinese (Simplified)"])
assert "Numbers kept" in failed_checks("take 2 tablets", "tome dos tabletas seis veces al día",
                                      "take two tablets six times a day", LANGUAGES["Spanish"])
assert "Numbers kept" not in failed_checks("take 1 tablet every 4 hours", "pran yon grenn chak 4 èdtan",
                                          "take one tablet every 4 hours", LANGUAGES["Haitian Creole"])
print("Review fixes: all four bug cases behave correctly.")

## 5 · The translation model
**NLLB-200** ("No Language Left Behind") is a free model from Meta that translates between 200
languages, including many that big apps serve poorly. For each instruction we ask it for its
top few translations (beam search), and we translate each one back into English for the checks.

In [ ]:
# 5 · Load the translation model ──────────────────────────────────────────
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
if DEVICE == "cuda":
    model = model.half()  # half precision: faster on GPU, same translations in practice
model = model.to(DEVICE).eval()


@torch.inference_mode()
def translate(texts, src_code, tgt_code, n_candidates=1, num_beams=4, batch_size=16):
    """Translate a list of texts. Returns, for each text, its top `n_candidates` translations,
    best first (the model's own ranking from beam search)."""
    tokenizer.src_lang = src_code
    num_beams = max(num_beams, n_candidates)
    results = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True,
                           max_length=256).to(DEVICE)
        output = model.generate(**inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_code),
                                num_beams=num_beams, num_return_sequences=n_candidates, max_length=256)
        decoded = tokenizer.batch_decode(output, skip_special_tokens=True)
        for i in range(len(batch)):
            results.append(decoded[i * n_candidates:(i + 1) * n_candidates])
    return results


example = "do not take more than 6 caplets in 24 hours"
for language in EVAL_LANGUAGES:
    print(f"{language}: {translate([example], 'eng_Latn', LANGUAGES[language]['code'])[0][0]}")

## 6 · Translate and check every instruction
For each instruction and language we keep two versions:
- **NLLB as-is** – the model's first choice (the baseline anyone would get).
- **Safer mode** – out of the model's top `N_CANDIDATES` translations, the one that fails the
  fewest checks on the **translation itself**, then the fewest checks overall (ties go to the
  model's own ranking). This is our improvement. We rank the translation-side checks first
  because a back-translation can lose a word like "under" even when the translation was right.

In [ ]:
# 6 · Run every instruction through both versions ─────────────────────────
def translate_and_check(texts, language, n_candidates):
    """Translate each English text, translate every candidate back, run the checks, and keep
    the model's first choice ("baseline") and the candidate that passes the most checks ("safer")."""
    spec = LANGUAGES[language]
    candidates = translate(texts, "eng_Latn", spec["code"], n_candidates=n_candidates)
    flat = [translation for options in candidates for translation in options]
    backs = [options[0] for options in translate(flat, spec["code"], "eng_Latn")]
    results, k = [], 0
    for source, options in zip(texts, candidates):
        scored = []
        for rank, translation in enumerate(options):
            checks = safety_checks(source, translation, backs[k], spec)
            scored.append({"rank": rank, "translation": translation, "back": backs[k], "checks": checks,
                           "failed": [name for name, c in checks.items() if not c["ok"]]})
            k += 1
        safer = min(scored, key=lambda option: (
            sum(name in FORWARD_CHECKS for name in option["failed"]),  # problems in the translation itself
            len(option["failed"]),                                     # then problems overall
            option["rank"]))                                           # then the model's own ranking
        results.append({"source": source, "baseline": scored[0], "safer": safer,
                        "changed": safer["rank"] != 0})
    return results


data["has_number"] = data.instruction.map(lambda t: any(number_parts(t, ENGLISH)))
data["has_not"] = data.instruction.map(lambda t: count_negations(t, ENGLISH) > 0)
data["has_limit"] = data.instruction.map(lambda t: bool(limit_groups(t, ENGLISH)))
data["risky"] = data.has_number | data.has_not | data.has_limit
risky, plain = data[data.risky], data[~data.risky]
n_risky = min(N_INSTRUCTIONS, len(risky))
sample_df = pd.concat([risky.sample(n_risky, random_state=0),
                       plain.sample(min(N_INSTRUCTIONS - n_risky, len(plain)), random_state=0)])
print(f"{data.risky.mean():.0%} of the {len(data)} instructions contain a number, a 'not' or an "
      f"age/amount limit. Testing {len(sample_df)} of them in {', '.join(EVAL_LANGUAGES)}.")

rows = []
for language in EVAL_LANGUAGES:
    started = time.time()
    results = translate_and_check(sample_df.instruction.tolist(), language, N_CANDIDATES)
    for instruction_id, result in zip(sample_df.instruction_id, results):
        rows.append({"language": language, "instruction_id": instruction_id, "source": result["source"],
                     "baseline": result["baseline"]["translation"], "baseline_back": result["baseline"]["back"],
                     "baseline_failed": result["baseline"]["failed"],
                     "safer": result["safer"]["translation"], "safer_back": result["safer"]["back"],
                     "safer_failed": result["safer"]["failed"], "changed": result["changed"]})
    print(f"{language}: {len(results)} instructions done in {time.time() - started:.0f} s")
results_df = pd.DataFrame(rows)
results_df.to_csv(DATA_DIR / "results.csv", index=False)

## 7 · Results from the automatic checks
⚠️ Safer mode picks translations *using* these checks, so it will always look better here.
These numbers show where the checks see problems; the real test is section 8.

In [ ]:
# 7 · Summary tables and examples ──────────────────────────────────────────
SYSTEMS = {"baseline": "NLLB as-is", "safer": f"Safer mode (best of {N_CANDIDATES})"}
summary = []
for language, group in results_df.groupby("language", sort=False):
    for system, label in SYSTEMS.items():
        failed = group[f"{system}_failed"]
        row = {"language": language, "version": label, "flagged %": 100 * failed.map(bool).mean()}
        for check in CHECK_NAMES:
            row[check] = 100 * failed.map(lambda names: check in names).mean()
        summary.append(row)
summary = pd.DataFrame(summary).round(1)
display(Markdown("**Share of translations failing each check (%)**"))
display(summary)

for language in EVAL_LANGUAGES:
    group = results_df[results_df.language == language]
    flagged = group[group.baseline_failed.map(bool)].head(5)
    display(Markdown(f"**{language}: translations the checks flagged** (NLLB as-is)"))
    display(pd.DataFrame({"English (label)": flagged.source, language: flagged.baseline,
                          "Back to English": flagged.baseline_back,
                          "Problems found": flagged.baseline_failed.map(", ".join)}).reset_index(drop=True))
    changed = group[group.changed].head(3)
    if len(changed):
        display(Markdown(f"**{language}: where safer mode picked a different translation**"))
        display(pd.DataFrame({"English (label)": changed.source, "NLLB as-is": changed.baseline,
                              "Safer mode": changed.safer,
                              "Safer mode, back to English": changed.safer_back}).reset_index(drop=True))

print("Numbers for the write-up:")
for language in EVAL_LANGUAGES:
    group = results_df[results_df.language == language]
    print(f"- {language}: checks flagged {group.baseline_failed.map(bool).mean():.0%} of NLLB's "
          f"translations; safer mode changed {group.changed.mean():.0%} of them and "
          f"{group.safer_failed.map(bool).mean():.0%} stayed flagged.")

## 8 · The real test: a fluent speaker rates the translations
The automatic checks can't understand meaning, so a teammate who reads `TEAM_LANGUAGE`
fluently rates translations **without knowing which version made them or whether they were
flagged**. Then we can measure honestly:
- how often each version is **dangerous**, and
- how many dangerous translations each checker **misses** (false negatives) and how many
  **false alarms** it raises (false positives): our full checks vs. a back-translation-only check.

**Steps:** run this cell → download `rate_me_<language>.csv` **and** `answer_key_<language>.csv`
from the Files panel (📁 on the left) → your teammate fills in the `rating` column (and
`correct_translation` for any dangerous one) → upload the filled sheet into `data/` named
`rated_<language>.csv` → run this cell again.

In [ ]:
# 8 · Rating sheet and evaluation ──────────────────────────────────────────
RATING_GUIDE = """Rating guide (column `rating`):
  2 = correct: a patient would follow it exactly right
  1 = small mistake, but the meaning and safety are the same
  0 = DANGEROUS: wrong number, dose, time or age limit; the "not" is lost; or confusing enough to be misused"""
slug = TEAM_LANGUAGE.split()[0].lower()
SHEET_FILE, KEY_FILE = DATA_DIR / f"rate_me_{slug}.csv", DATA_DIR / f"answer_key_{slug}.csv"
RATED_FILE = DATA_DIR / f"rated_{slug}.csv"


def make_rating_sheet(language, max_instructions=60):
    group = results_df[results_df.language == language].head(max_instructions)
    items = []
    for _, r in group.iterrows():
        versions = [("baseline", r.baseline, r.baseline_failed)]
        if r.changed:
            versions.append(("safer", r.safer, r.safer_failed))
        for system, translation, failed in versions:
            items.append({"instruction_id": r.instruction_id, "system": system, "english": r.source,
                          "translation": translation, "flag_full": bool(failed),
                          "flag_backonly": "Back-translation matches" in failed})
    items = pd.DataFrame(items).sample(frac=1, random_state=1).reset_index(drop=True)  # shuffle = blind
    items.insert(0, "item_id", range(1, len(items) + 1))
    sheet = items[["item_id", "english", "translation"]].assign(rating="", correct_translation="", notes="")
    return sheet, items[["item_id", "instruction_id", "system", "english", "translation", "flag_full", "flag_backonly"]]


def checker_scores(df, column):
    flagged, dangerous = df[column].astype(bool), df.dangerous
    tp, fn = int((flagged & dangerous).sum()), int((~flagged & dangerous).sum())
    fp, tn = int((flagged & ~dangerous).sum()), int((~flagged & ~dangerous).sum())
    return {"dangerous caught": tp, "dangerous MISSED (false negatives)": fn,
            "false alarms (false positives)": fp, "correctly passed": tn,
            "catch rate %": round(100 * tp / (tp + fn), 1) if tp + fn else None,
            "false alarm rate %": round(100 * fp / (fp + tn), 1) if fp + tn else None}


def evaluate_ratings(rated_file, key_file):
    rated, key = pd.read_csv(rated_file), pd.read_csv(key_file)
    df = key.merge(rated[["item_id", "english", "translation", "rating"]], on="item_id", suffixes=("", "_rated"))
    mismatched = (df.english != df.english_rated) | (df.translation != df.translation_rated)
    if mismatched.any():
        raise ValueError(f"{int(mismatched.sum())} rated rows don't match the answer key (item ids point at other "
                         "translations). Use the answer key that was made with this rating sheet.")
    df["rating"] = pd.to_numeric(df.rating, errors="coerce")
    df = df.dropna(subset=["rating"])
    df["dangerous"] = df.rating == 0
    print(f"{len(df)} rated translations.\n")

    per_version = {"baseline": [], "safer": []}
    for _, group in df.groupby("instruction_id"):
        base = group[group.system == "baseline"]
        if base.empty:
            continue
        safer = group[group.system == "safer"]
        per_version["baseline"].append(bool(base.dangerous.iloc[0]))
        per_version["safer"].append(bool((safer if len(safer) else base).dangerous.iloc[0]))
    display(Markdown("**How often is each version dangerous?**"))
    display(pd.DataFrame([{"version": SYSTEMS[s], "instructions rated": len(v),
                           "dangerous %": round(100 * sum(v) / len(v), 1) if v else None}
                          for s, v in per_version.items()]))

    display(Markdown("**Does each checker catch the dangerous translations?**"))
    display(pd.DataFrame([{"checker": "Back-translation only (what people usually do)",
                           **checker_scores(df, "flag_backonly")},
                          {"checker": "Our full safety checks", **checker_scores(df, "flag_full")}]))
    return df


if RATED_FILE.exists() and KEY_FILE.exists():
    rated_df = evaluate_ratings(RATED_FILE, KEY_FILE)
elif KEY_FILE.exists():
    print(f"An answer key already exists ({KEY_FILE}), so no new sheet was made: a new one would no longer match")
    print(f"the sheet your rater has. Upload the filled sheet as {RATED_FILE}, or delete the key to start over.")
else:
    sheet, key = make_rating_sheet(TEAM_LANGUAGE)
    sheet.to_csv(SHEET_FILE, index=False)
    key.to_csv(KEY_FILE, index=False)
    print(f"Made a rating sheet with {len(sheet)} translations: {SHEET_FILE}")
    print(f"Answer key (don't show the rater): {KEY_FILE}\n")
    print(RATING_GUIDE)
    print(f"\nWhen it's filled in, upload it as {RATED_FILE} and run this cell again.")

## 9 · (Optional) General translation quality on FLORES+
FLORES+ is a standard test set: the same 1,000 sentences professionally translated into 200+
languages. Scoring NLLB on it shows how much worse it is in general for less widely served
languages. It needs a free Hugging Face account: accept the terms at
https://huggingface.co/datasets/openlanguagedata/flores_plus, create an access token, and add
it in Colab under 🔑 **Secrets** with the name `HF_TOKEN`. Skipped automatically otherwise.

In [ ]:
# 9 · FLORES+ chrF score (optional) ────────────────────────────────────────
def hugging_face_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None


def flores_sentences(codes, n, token):
    from datasets import load_dataset
    last_error = None
    for code in codes:
        try:
            return load_dataset("openlanguagedata/flores_plus", code, split="devtest", token=token)["text"][:n]
        except Exception as error:
            last_error = error
    raise last_error


token = hugging_face_token()
if QUICK or not token:
    print("Skipped: " + ("quick test mode." if QUICK else "no Hugging Face token (see the note above)."))
else:
    ensure_packages([("datasets", "datasets")])
    english = flores_sentences(["eng_Latn"], 100, token)
    for language in EVAL_LANGUAGES:
        try:
            references = flores_sentences(LANGUAGES[language]["flores"], 100, token)
        except Exception as error:
            print(f"{language}: couldn't load FLORES+ ({error})")
            continue
        outputs = [options[0] for options in translate(english, "eng_Latn", LANGUAGES[language]["code"])]
        print(f"{language}: chrF {sacrebleu.corpus_chrf(outputs, [references]).score:.1f} / 100")

## 10 · Photo input: reading the label from a photo
Instead of typing, the pharmacist can **photograph the Directions section** of the label.
**EasyOCR**, a free text-reading model, finds each word in the photo, reads it, and says how sure
it is. Reading a photo adds a new way to make a dangerous mistake: if *"1/2 teaspoon"* is read as
*"12 teaspoon"*, every translation check still passes, because the checks compare the translation
with the text that came out of the photo. So the app
- **highlights every word the OCR wasn't sure about**, with a stricter bar for numbers
  (`NUMBER_CONFIDENCE_MIN`) than for other words (`OCR_CONFIDENCE_MIN`), plus letters mixed into
  a number, like *"l2"*, and
- puts the text it read in a box the pharmacist **checks against the label** before translating.

Two things we found while building it: with EasyOCR's default settings, one-character words like
the *4* in *"3 to 4 times"* were silently dropped, and the ■ bullet was often read as a number
(*"■ do not take…"* became *"2 do not take…"*). We fixed both (see `read_photo`).

**How we test it.** We don't have real label photos with their correct text, so we make our own:
each test instruction is printed like a label (with a ■ bullet), then made to look like a phone
photo (tilted, blurred, unevenly lit, noisy and JPEG-compressed) at four levels of blur. Because
we know the true text, we can count **dangerous misreads** (a number, a "not" or an under/over
limit changed) and how many of them the highlighting catches.

In [ ]:
# 10 · Read a photo of the label, and test it on made-up label photos ──────
import difflib
import io
import textwrap
import zlib

import matplotlib
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont, ImageOps

# Words read with less confidence (0–1) than this are highlighted. We're stricter with numbers,
# because a wrong number is the most dangerous misread. We picked these two values on made-up
# photos of 60 label instructions that are NOT in the test below.
OCR_CONFIDENCE_MIN = 0.3
NUMBER_CONFIDENCE_MIN = 0.7
try:
    ensure_packages([("easyocr", "easyocr")])
    import easyocr
    ocr_reader = easyocr.Reader(["en"], gpu=DEVICE == "cuda", verbose=False)
except Exception as error:  # the app still works by typing if the photo reader can't start
    ocr_reader = None
    print(f"Photo input is off: EasyOCR couldn't start ({error}). Typing still works.")

HEADINGS = ["drug facts (continued)", "drug facts", "directions", "uses", "warnings",
            "other information", "inactive ingredients"]  # section titles, left out of the text
STARTERS = ("adults", "adult", "children", "child", "teens", "teenagers", "persons", "infants",
            "do not", "don't")  # a line that starts like this is a new instruction (a new table row)
CONNECTORS = {"and", "or", "to", "of", "than", "the", "a", "an", "every", "by", "for", "in", "with",
              "under", "over", "if", "not"}  # a line that ends like this continues on the next line
BULLET = re.compile(r"^[■□▪▫●•◦‣∙*=»]+")     # OCR often reads the ■ bullet as = or •
BORDER = re.compile(r"^[|\[\]{}_]+$")        # table borders read as characters
MIXED = re.compile(r"(?=.*\d)(?=.*[a-z])", re.I)
UNIT_SUFFIX = re.compile(r"^\d+(?:[.,/]\d+)?(?:mg|mcg|ml|g|oz|hrs?|h|th|st|nd|rd|x|yrs?)$", re.I)


def read_photo(image):
    """Read every word in a photo. Returns its lines from top to bottom; each line is a list of
    (word, confidence) pairs from left to right."""
    pixels = np.asarray(image.convert("RGB"))
    gray = pixels.mean(axis=2)
    # One box per word (width_ths=0), and keep small boxes (min_size=4): with EasyOCR's defaults,
    # one-character words like the "4" in "3 to 4 times" were silently dropped in our tests.
    found = ocr_reader.readtext(pixels, width_ths=0.0, min_size=4)
    boxes = []
    for corners, word, confidence in found:
        xs, ys = [float(x) for x, _ in corners], [float(y) for _, y in corners]
        left, right, top, bottom = min(xs), max(xs), min(ys), max(ys)
        word, confidence = word.strip(), float(confidence)
        # The ■ bullet is often read as "2", "4" or "E", which would add a number to the
        # instruction. A box that is roughly square and mostly ink is a bullet, not a character.
        patch = gray[int(max(top, 0)):int(bottom) + 1, int(max(left, 0)):int(right) + 1]
        if patch.size and 0.6 < (right - left) / max(1.0, bottom - top) < 1.6:
            if (patch < (float(patch.min()) + float(patch.max())) / 2).mean() > 0.5:
                word, confidence = "■", 1.0
        if word and not BORDER.match(word):
            boxes.append({"left": left, "top": top, "bottom": bottom, "word": word, "confidence": confidence})
    lines = []
    for box in sorted(boxes, key=lambda b: b["top"] + b["bottom"]):
        middle = (box["top"] + box["bottom"]) / 2
        if lines and lines[-1][-1]["top"] <= middle <= lines[-1][-1]["bottom"]:
            lines[-1].append(box)  # same row as the word before it
        else:
            lines.append([box])
    return [[(b["word"], b["confidence"]) for b in sorted(line, key=lambda b: b["left"])]
            for line in lines]


def photo_warning(word, confidence, number_min=None):
    """Why a word read from a photo should be checked against the label (None if it looks fine)."""
    has_digit = any(char.isdigit() for char in word)
    minimum = (NUMBER_CONFIDENCE_MIN if number_min is None else number_min) if has_digit else OCR_CONFIDENCE_MIN
    if confidence < minimum:
        return f"“{word}” ({confidence:.0%} sure)"
    for token in re.split(r"[-–/]", word.strip(",.;:()")):
        if MIXED.search(token) and not UNIT_SUFFIX.match(token):
            return f"“{word}” (letters mixed into a number)"
    return None


def photo_to_instructions(image):
    """Read a label photo and join its lines into whole instructions, one per bullet or table row.
    Each instruction keeps the words it was read from and the ones to check."""
    chunks = []
    for line in read_photo(image):
        joined = "".join(word for word, _ in line)
        if sum(char.isdigit() for char in joined) >= 8 and not re.search(r"[^\W\d_]", joined):
            # The number printed under a barcode (no letters, 8+ digits): it is not part of the
            # instruction above it, so it ends that instruction and is left out.
            chunks.append(None)
            continue
        previous = chunks[-1] if chunks else None
        last = previous["words"][-1][0] if previous and previous["words"] else ""
        after_colon = last.endswith(":")  # "children under 2 years:" goes on to the next line
        plain = [clean(word).lower() for word, _ in line]
        for heading in HEADINGS:  # a capitalized section title like "Directions" is left out
            size = len(heading.split())
            if not after_colon and line[0][0][:1].isupper() and plain[:size] == heading.split():
                line, previous, last = line[size:], None, ""
                chunks.append(None)  # a title also ends the instruction before it
                break
        if not line:
            continue
        first_words = clean(" ".join(word for word, _ in line[:3])).lower()
        line_starts_new = (not previous or not previous["words"] or (not after_colon and (
            last.endswith((".", ";")) or
            (first_words.startswith(STARTERS) and clean(last).lower() not in CONNECTORS))))
        for i, (word, confidence) in enumerate(line):
            bullet = BULLET.match(word)
            word = BULLET.sub("", word).strip()
            if bullet or (i == 0 and line_starts_new):
                chunks.append({"words": []})
            if word:
                chunks[-1]["words"].append((word, confidence))
    instructions = []
    for chunk in chunks:
        if not chunk or not chunk["words"]:
            continue
        text = clean(" ".join(word for word, _ in chunk["words"]))
        if len(text.split()) < 2 and not re.search(r"\d", text):
            continue  # a stray mark, not an instruction
        unsure = [(word, warning) for word, confidence in chunk["words"]
                  if (warning := photo_warning(word, confidence))]
        instructions.append({"text": text, "words": chunk["words"], "unsure": unsure})
    return instructions


# Made-up label photos, so we know the true text
FONT_DIR = Path(matplotlib.get_data_path()) / "fonts" / "ttf"
PHOTO_STYLES = {  # how much each made-up photo is spoiled: more blur also means more tilt, noise and JPEG loss
    name: {"blur": blur, "tilt": 1 + blur, "noise": 4 + 6 * blur, "jpeg": int(90 - 30 * blur)}
    for name, blur in [("sharp", 0.3), ("slightly blurry", 0.9), ("blurry", 1.2), ("very blurry", 1.5)]}


def draw_label(instructions, heading=None, font_size=24, width=38):
    """Print instructions the way a Drug Facts label does: a bold heading, then ■ bullets."""
    font = ImageFont.truetype(str(FONT_DIR / "DejaVuSans.ttf"), font_size)
    bold = ImageFont.truetype(str(FONT_DIR / "DejaVuSans-Bold.ttf"), font_size + 4)
    rows = [(heading, bold)] if heading else []
    for text in instructions:
        wrapped = textwrap.wrap(text, width)
        rows += [("■ " + wrapped[0], font)] + [("    " + part, font) for part in wrapped[1:]]
    step, margin = int(font_size * 1.45), font_size
    image = Image.new("L", (max(int(f.getlength(t)) for t, f in rows) + 2 * margin,
                            step * len(rows) + 2 * margin), 248)
    draw = ImageDraw.Draw(image)
    for i, (text, f) in enumerate(rows):
        draw.text((margin, margin + i * step), text, fill=25, font=f)
    return image


def spoil_photo(image, style, rng):
    """Make a clean label image look like a quick phone photo."""
    image = image.rotate(rng.uniform(-style["tilt"], style["tilt"]), resample=Image.BICUBIC,
                         expand=True, fillcolor=248)
    image = image.filter(ImageFilter.GaussianBlur(style["blur"]))
    pixels = np.asarray(image, dtype=np.float32)
    light = np.linspace(rng.uniform(0.7, 1.0), rng.uniform(0.9, 1.05), pixels.shape[1])  # uneven light
    pixels = pixels * light[None, :] + rng.normal(0, style["noise"], pixels.shape)
    buffer = io.BytesIO()
    Image.fromarray(np.clip(pixels, 0, 255).astype(np.uint8)).save(buffer, "JPEG", quality=style["jpeg"])
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")


def photo_rng(text, style_name):
    """The same sentence and style always get the same made-up photo, so editing one sentence
    in the data doesn't change every other photo (and the results) too."""
    return np.random.default_rng([zlib.crc32(text.encode("utf-8")), zlib.crc32(style_name.encode("utf-8"))])


def misread_problems(source, text):
    """What a misread changed that could hurt someone: a number, a "not" or an under/over limit."""
    problems = []
    ok, detail = compare_numbers(source, text, ENGLISH)
    if not ok:
        problems.append("numbers " + detail)
    if count_negations(text, ENGLISH) != count_negations(source, ENGLISH):
        problems.append("“not” changed")
    lost = limit_groups(source, ENGLISH) - limit_groups(text, ENGLISH)
    if lost:
        problems.append("lost " + ", ".join(sorted(lost)))
    return problems


def comparable(text):
    return " ".join(re.sub(r"[^a-z0-9/]+", " ", normalize(text)).split())


def edit_distance(a, b):
    previous = list(range(len(b) + 1))
    for i, char_a in enumerate(a, 1):
        current = [i]
        for j, char_b in enumerate(b, 1):
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + (char_a != char_b)))
        previous = current
    return previous[-1]


def highlighted(words, number_min=None):
    return any(photo_warning(word, confidence, number_min) for word, confidence in words)


EXAMPLE_DIR = DATA_DIR / "example_photos"
EXAMPLE_DIR.mkdir(exist_ok=True)
EXAMPLE_PHOTOS = []
if ocr_reader is not None:
    photo_test = sample_df[sample_df.risky].instruction.head(15 if QUICK else 100).tolist()
    rows, examples, started = [], {}, time.time()
    for style_name, style in PHOTO_STYLES.items():
        for source in photo_test:
            photo = spoil_photo(draw_label([source]), style, photo_rng(source, style_name))
            found = photo_to_instructions(photo)
            text = " ".join(item["text"] for item in found)
            rows.append({"style": style_name, "source": source, "read": text,
                         "words": [pair for item in found for pair in item["words"]],
                         "exact": comparable(text) == comparable(source),
                         "wrong characters": edit_distance(comparable(text), comparable(source))
                                             / max(1, len(comparable(source))),
                         "problems": misread_problems(source, text)})
            examples.setdefault(style_name, photo)
    photo_df = pd.DataFrame(rows)
    photo_df["dangerous"] = photo_df.problems.map(bool)
    print(f"Read {len(photo_df)} made-up label photos in {time.time() - started:.0f} s")

    summary_rows = []
    for style_name, group in photo_df.groupby("style", sort=False):
        flags, dangerous = group.words.map(highlighted), group.dangerous
        summary_rows.append({"photo": style_name, "instructions": len(group),
                             "read exactly %": round(100 * group.exact.mean(), 1),
                             "characters wrong %": round(100 * group["wrong characters"].mean(), 1),
                             "dangerous misreads": int(dangerous.sum()),
                             "caught by the highlight": int((flags & dangerous).sum()),
                             "MISSED (false negatives)": int((~flags & dangerous).sum()),
                             "false alarms (false positives)": int((flags & ~dangerous).sum())})
    display(Markdown("**Reading made-up label photos** (a dangerous misread changes a number, a "
                     "“not” or an under/over limit)"))
    display(pd.DataFrame(summary_rows))

    tradeoff = []
    for number_min in [OCR_CONFIDENCE_MIN, 0.5, 0.7, 0.9]:
        flags, dangerous = photo_df.words.map(lambda w: highlighted(w, number_min)), photo_df.dangerous
        tradeoff.append({"highlight numbers read below": number_min,
                         "dangerous misreads caught %": round(100 * (flags & dangerous).sum()
                                                              / max(1, dangerous.sum()), 1),
                         "false alarm rate %": round(100 * (flags & ~dangerous).sum()
                                                     / max(1, (~dangerous).sum()), 1)})
    display(Markdown("**Trade-off:** being stricter with numbers catches more misreads but raises more "
                     f"false alarms (the app uses {NUMBER_CONFIDENCE_MIN} for numbers and "
                     f"{OCR_CONFIDENCE_MIN} for other words)"))
    display(pd.DataFrame(tradeoff))

    misreads = photo_df[photo_df.dangerous].copy()
    misreads["highlighted?"] = misreads.words.map(lambda w: "yes" if highlighted(w) else "NO")
    misreads = misreads.sort_values("highlighted?", kind="stable").head(8)  # missed ones ("NO") first
    if len(misreads):
        display(Markdown("**Where it fails: dangerous misreads** (the ones the highlight missed come first)"))
        display(pd.DataFrame({"photo": misreads["style"], "label says": misreads.source,
                              "OCR read": misreads.read, "what changed": misreads.problems.map("; ".join),
                              "highlighted?": misreads["highlighted?"]}).reset_index(drop=True))
    for style_name, photo in examples.items():
        print(f"Example of a made-up photo ({style_name}):")
        display(photo)

    # Example photos for the app: a Directions panel from one real label
    directions = data[data.section == "Directions"]
    panels = [group for _, group in directions.groupby("set_id", sort=False)
              if 3 <= len(group) <= 5 and group.instruction.str.len().max() < 100]
    if panels:  # pick a label whose directions mix numbers, "not" and age limits
        panel = max(panels, key=lambda g: (g.has_not.any() + g.has_limit.any() + g.has_number.mean(),
                                           g.instruction.str.split().str[0].str.lower().nunique())
                    ).instruction.tolist()
        for name in ["sharp", "slightly blurry"]:
            path = EXAMPLE_DIR / f"directions_{name.replace(' ', '_')}.jpg"
            spoil_photo(draw_label(panel, heading="Directions"), PHOTO_STYLES[name],
                        photo_rng(" ".join(panel), name)).save(path)
            EXAMPLE_PHOTOS.append(path)
        read_back = [item["text"] for item in photo_to_instructions(Image.open(EXAMPLE_PHOTOS[0]))]
        print(f"Example label: printed {len(panel)} instructions, read back {len(read_back)}:")
        display(pd.DataFrame({"printed": pd.Series(panel), "read back": pd.Series(read_back)}))

## 11 · Try it: the Dose-Safe app
**Type** an instruction, or take a **photo** of the label's Directions section. Choose a language
and see the translation, the back-translation and every check. The app prints a public link ending
in `.gradio.live` that also works on a phone, while this notebook is running.
**Prototype – not medical advice.**

In [ ]:
# 11 · Gradio demo app ─────────────────────────────────────────────────────
import gradio as gr


def check_one(english, language, safer_mode):
    english = (english or "").strip()
    if not english:
        return "", "", "Type a medicine instruction in English first."
    result = translate_and_check([english], language, N_CANDIDATES if safer_mode else 1)[0]
    chosen = result["safer"] if safer_mode else result["baseline"]
    lines = []
    for name, check in chosen["checks"].items():
        lines.append(f"{'✅' if check['ok'] else '❌'} **{name}**" + ("" if check["ok"] else f" ({check['detail']})"))
    if chosen["failed"]:
        verdict = "### ⚠️ Ask a pharmacist or interpreter to check this translation"
    else:
        verdict = ("### ✅ No problems found by the automatic checks\n"
                   "A pharmacist should still confirm before giving it to a patient.")
    return chosen["translation"], chosen["back"], verdict + "\n\n" + "\n\n".join(lines)


def read_label(photo_path):
    """Photo step 1: read the photo, and show which words to check against the label."""
    if ocr_reader is None:
        return [], "", "Photo input is off because EasyOCR couldn't start. Typing still works.", []
    if not photo_path:
        return [], "", "Take or upload a photo of the label first.", []
    found = photo_to_instructions(ImageOps.exif_transpose(Image.open(photo_path)))
    if not found:
        return [], "", "⚠️ No text found. Try a closer, sharper photo of the Directions section.", []
    marked = []
    for item in found:
        for word, confidence in item["words"]:
            marked.append((word + " ", "check this" if photo_warning(word, confidence) else None))
        marked.append(("\n", None))
    n_unsure = sum(len(item["unsure"]) for item in found)
    notes = f"Read **{len(found)} instructions**. " + (
        f"⚠️ **{n_unsure} word(s) were hard to read** (highlighted). Compare them with the label and fix "
        "the text in the box before translating." if n_unsure else
        "No word was hard to read, but still compare the text with the label.")
    return marked, "\n".join(item["text"] for item in found), notes, found


def still_unsure(line, found):
    """Hard-to-read words from the photo that are still in this line (after any corrections)."""
    if not found:
        return []
    best = max(found, key=lambda item: difflib.SequenceMatcher(None, item["text"], line).ratio())
    if difflib.SequenceMatcher(None, best["text"], line).ratio() < 0.6:
        return []
    return [warning for word, warning in best["unsure"]
            if re.search(rf"(?<!\w){re.escape(word)}(?!\w)", line)]


def translate_photo_text(text, language, safer_mode, found):
    """Photo step 2: translate and check every line in the box."""
    lines = [line.strip() for line in (text or "").splitlines() if line.strip()]
    if not lines:
        return None, "Read a photo first (or type instructions in the box, one per line)."
    results = translate_and_check(lines, language, N_CANDIDATES if safer_mode else 1)
    rows = []
    for line, result in zip(lines, results):
        chosen = result["safer"] if safer_mode else result["baseline"]
        problems = ["photo hard to read: " + warning for warning in still_unsure(line, found)]
        problems += [f"{name} ({chosen['checks'][name]['detail']})" for name in chosen["failed"]]
        rows.append({"Verdict": "⚠️ check" if problems else "✅", "English (from the photo)": line,
                     "Translation": chosen["translation"], "Why": "; ".join(problems),
                     "Back to English": chosen["back"]})
    n_flagged = sum(row["Verdict"] != "✅" for row in rows)
    summary = (f"### ⚠️ Ask a pharmacist or interpreter to check {n_flagged} of these {len(rows)} lines"
               if n_flagged else "### ✅ No problems found by the automatic checks\n"
               "A pharmacist should still confirm before giving them to a patient.")
    return pd.DataFrame(rows), summary


demo_examples = sample_df[sample_df.risky].instruction.head(6).tolist()
with gr.Blocks(title="Dose-Safe Translator") as demo:
    gr.Markdown("# 💊 Dose-Safe Translator\nTranslates over-the-counter medicine directions and flags "
                "translations a pharmacist should double-check.\n\n*Prototype – not medical advice.*")
    with gr.Row():
        language_box = gr.Dropdown(list(LANGUAGES), value=TEAM_LANGUAGE, label="Translate to")
        safer_box = gr.Checkbox(value=True, label=f"Safer mode: try {N_CANDIDATES} translations "
                                                  "and keep the one that passes the most checks")
    with gr.Tab("⌨️ Type an instruction"):
        with gr.Row():
            with gr.Column():
                english_box = gr.Textbox(label="Medicine instruction (English)", lines=3,
                                         value="do not take more than 6 caplets in 24 hours")
                run_button = gr.Button("Translate and check", variant="primary")
                gr.Examples(examples=[[e] for e in demo_examples], inputs=[english_box])
            with gr.Column():
                translation_box = gr.Textbox(label="Translation", lines=3)
                back_box = gr.Textbox(label="Translated back into English (for checking)", lines=3)
                report_box = gr.Markdown()
    with gr.Tab("📷 Photo of the label"):
        if ocr_reader is None:
            gr.Markdown("Photo input is off because EasyOCR couldn't start. Typing still works.")
        found_state = gr.State([])
        with gr.Row():
            with gr.Column():
                photo_box = gr.Image(label="Photo of the Directions section (flat, close and in focus)",
                                     type="filepath", sources=["upload", "webcam", "clipboard"])
                read_button = gr.Button("1 · Read the label")
                if EXAMPLE_PHOTOS:
                    gr.Examples(examples=[[str(path)] for path in EXAMPLE_PHOTOS], inputs=[photo_box],
                                label="Made-up example labels")
            with gr.Column():
                photo_notes = gr.Markdown("Take or upload a photo, then click **1 · Read the label**.")
                marked_box = gr.HighlightedText(label="What the app read (highlighted = hard to read)",
                                                color_map={"check this": "orange"})
                photo_text_box = gr.Textbox(label="Check this against the label and fix any mistakes "
                                                  "(one instruction per line)", lines=6)
                translate_all_button = gr.Button("2 · Translate and check every line", variant="primary")
        photo_summary = gr.Markdown()
        photo_table = gr.Dataframe(headers=["Verdict", "English (from the photo)", "Translation", "Why",
                                            "Back to English"], wrap=True, interactive=False,
                                   column_widths=["10%", "22%", "24%", "22%", "22%"])
    run_button.click(check_one, inputs=[english_box, language_box, safer_box],
                     outputs=[translation_box, back_box, report_box])
    read_button.click(read_label, inputs=[photo_box],
                      outputs=[marked_box, photo_text_box, photo_notes, found_state])
    translate_all_button.click(translate_photo_text,
                               inputs=[photo_text_box, language_box, safer_box, found_state],
                               outputs=[photo_table, photo_summary])

if not QUICK:
    # share=True gives a public https://….gradio.live link: it opens in any browser or phone (the
    # localhost link Colab shows only works inside Colab). It stops when this notebook stops.
    demo.launch(share=True, debug=False, allowed_paths=[str(EXAMPLE_DIR.resolve())])